 # Perturbation Analysis



 scHopfield supports two complementary strategies for in-silico perturbations:



 **ODE-based** — integrates the full dynamical system

 $$\frac{dx}{dt} = W \cdot \sigma(x) - \gamma \cdot x + I$$

 with a fixed perturbation (e.g. Gata1 = 0 for knockout).



 **GRN propagation** — propagates expression shifts through the learned network

 without full time integration (analogous to the CellOracle approach).



 This notebook uses the Paul et al. 2015 hematopoiesis dataset loaded through

 the CellOracle tutorial object, which provides a graph-based embedding and

 pseudotime ordering.



 Topics covered:

 1. Data loading, velocity estimation, sigmoid fitting, GRN inference

 2. ODE-based single-cell trajectories (WT, KO, OE)

 3. Dataset-wide ODE perturbation shift

 4. GRN propagation-based perturbation

 5. Perturbation flow on embedding (CellOracle-style and Hopfield-style)

 ## 5.1 Setup

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import torch

import celloracle as co
import scHopfield as sch

warnings.filterwarnings('default')

# Analysis parameters
CLUSTER_KEY    = 'paul15_clusters'
SPLICED_KEY    = 'Ms'
DEGRADATION_KEY = 'gamma'
BASIS          = 'draw_graph_fa'
VELOCITY_KEY   = 'velocity_S'
VELOCITY_SCALE = 500.0
SCAFFOLD_REG   = 1e-1
N_EPOCHS       = 1000
BATCH_SIZE     = 128
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

# Gene of interest for perturbation
GOI = 'Gata1'

# Paul15 cluster display order
CLUSTER_ORDER = [
    '1Ery', '2Ery', '3Ery', '4Ery', '5Ery', '6Ery', '7MEP',
    '8Mk', '9Mk', '9GMP', '10GMP', '11DC', '12Baso', '13Baso',
    '14Mo', '15Mo', '16Neu', '17Neu', '18Eos', '19Lymph',
]

print(f"Device: {DEVICE}")


/home/bernaljp/miniconda3/envs/SCH/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
which: no R in (/opt/slurm/puppet/bin:/opt/slurm/cluster/ibex/install-v2/RedHat-9/bin:/opt/slurm/scripts/bin:/usr/lpp/mmfs/bin:/home/bernaljp/miniconda3/envs/SCH/bin:/opt/slurm/puppet/bin:/opt/slurm/cluster/ibex/install-v2/RedHat-9/bin:/opt/slurm/scripts/bin:/usr/lpp/mmfs/bin:/home/bernaljp/miniconda3/condabin:/opt/slurm/puppet/bin:/usr/share/Modules/bin:/opt/slurm/cluster/ibex/install-v2/RedHat-9/bin:/opt/slurm/scripts/bin:/usr/lpp/mmfs/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin:/opt/slurm/scripts/bin:/opt/puppetlabs/bin:/home/bernaljp/.local/bin:/home/bernaljp/bin:/opt/slurm/scripts/

Device: cuda


 ### Load CellOracle tutorial data (Paul et al. 2015)

In [2]:
oracle_demo = co.data.load_tutorial_oracle_object()
adata = oracle_demo.adata.copy()
print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"Cluster key: {CLUSTER_KEY}")
print(f"Cell types: {sorted(adata.obs[CLUSTER_KEY].unique())}")


Loaded: 2671 cells × 1999 genes
Cluster key: paul15_clusters
Cell types: ['10GMP', '11DC', '12Baso', '13Baso', '14Mo', '15Mo', '16Neu', '17Neu', '18Eos', '19Lymph', '1Ery', '2Ery', '3Ery', '4Ery', '5Ery', '6Ery', '7MEP', '8Mk', '9GMP']


 ### Estimate velocities from pseudotime

In [3]:
# Prepare moments (scVelo)
adata.layers['spliced'] = adata.layers['normalized_count']
adata.layers['unspliced'] = adata.layers['normalized_count']
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
_ = adata.layers.pop('unspliced')

# Estimate velocities from pseudotime
sch.pp.estimate_velocity_from_pseudotime(
    adata,
    pseudotime_key='Pseudotime',
    spliced_key=SPLICED_KEY,
    connectivity_key='connectivities',
    scale=VELOCITY_SCALE,
    store_key=VELOCITY_KEY,
)

# Compute velocity graph and embedding for later plotting
scv.tl.velocity_graph(adata, vkey=VELOCITY_KEY, xkey=SPLICED_KEY, n_jobs=-1)
scv.tl.velocity_embedding(adata, basis=BASIS, vkey=VELOCITY_KEY)
adata.obsm[f'velocity_{BASIS}'] = adata.obsm[f'{VELOCITY_KEY}_{BASIS}']

print(f"Velocity estimated with scale={VELOCITY_SCALE}.")


computing neighbors
    finished (0:00:06) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocity graph (using 32/32 cores)


  0%|          | 0/2671 [00:00<?, ?cells/s]

/home/bernaljp/miniconda3/envs/SCH/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/home/bernaljp/miniconda3/envs/SCH/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/home/bernaljp/miniconda3/envs/SCH/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The p

    finished (0:00:24) --> added 
    'velocity_S_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_S_draw_graph_fa', embedded velocity vectors (adata.obsm)
Velocity estimated with scale=500.0.


In [4]:
# Estimate degradation rates (gamma) from velocity / expression ratio
expression = adata.layers[SPLICED_KEY].copy()
velocities  = adata.layers[VELOCITY_KEY]

mean_expr = np.abs(expression).mean(axis=0) + 1e-6
mean_vel  = np.abs(velocities).mean(axis=0)
gamma     = np.clip(mean_vel / mean_expr, 0.1, 10.0)
adata.var[DEGRADATION_KEY] = gamma

print(f"Gamma range: [{gamma.min():.3f}, {gamma.max():.3f}]  |  median={np.median(gamma):.3f}")


Gamma range: [0.100, 0.100]  |  median=0.100


 ### Sigmoid fitting

In [5]:
adata.var['scHopfield_used'] = True  # use all genes

sch.pp.fit_all_sigmoids(
    adata,
    genes=adata.var['scHopfield_used'].values,
    spliced_key=SPLICED_KEY,
)
sch.pp.compute_sigmoid(adata, spliced_key=SPLICED_KEY)

mse = adata.var.loc[adata.var['scHopfield_used'], 'sigmoid_mse']
print(f"Sigmoid MSE: mean={mse.mean():.4f}, median={mse.median():.4f}")


/home/bernaljp/packages/scHopfield/scHopfield/_utils/math.py:93: RuntimeWarning: divide by zero encountered in divide
  ty = np.log(y / (1 - y))
/home/bernaljp/packages/scHopfield/scHopfield/_utils/math.py:93: RuntimeWarning: divide by zero encountered in log
  ty = np.log(y / (1 - y))


Sigmoid MSE: mean=0.0025, median=0.0016


In [6]:
# Plot sigmoid fits for key regulatory genes
genes_to_plot = [g for g in ['Gata1', 'Klf1', 'Gata2', 'Spi1', 'Cebpa', 'Mpo']
                 if g in adata.var_names]

n_cols = min(3, len(genes_to_plot))
n_rows = (len(genes_to_plot) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), tight_layout=True)
for i, gene in enumerate(genes_to_plot):
    sch.pl.plot_sigmoid_fit(adata, gene=gene, ax=np.atleast_2d(axes).flatten()[i],
                            spliced_key=SPLICED_KEY, show_zeros=False)
for j in range(i + 1, n_rows * n_cols):
    np.atleast_2d(axes).flatten()[j].axis('off')
plt.suptitle('Sigmoid fits for key regulatory genes', y=1.02)
plt.show()


 ### GRN scaffold from CellOracle base GRN

In [7]:
base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()
base_GRN.drop(['peak_id'], axis=1, inplace=True)

gene_names = adata.var.index[adata.var['scHopfield_used'].values]
scaffold = pd.DataFrame(0, index=gene_names, columns=gene_names)

tfs = list(set(base_GRN.columns.str.lower()) & set(scaffold.index.str.lower()))
targets = list(set(base_GRN['gene_short_name'].str.lower().values) & set(scaffold.columns.str.lower()))
index_map = {g.lower(): g for g in scaffold.index}
col_map   = {g.lower(): g for g in scaffold.columns}

for tf in tfs:
    tf_col = [c for c in base_GRN.columns if c.lower() == tf][0]
    for tgt in base_GRN[base_GRN[tf_col] == 1]['gene_short_name']:
        if tgt.lower() in col_map:
            scaffold.loc[index_map[tf], col_map[tgt.lower()]] = 1

print(f"Scaffold: {len(tfs)} TFs, {len(targets)} targets, {int(scaffold.sum().sum())} potential edges")


Scaffold: 90 TFs, 1857 targets, 75325 potential edges


 ### GRN inference

In [8]:
sch.inf.fit_interactions(
    adata,
    cluster_key=CLUSTER_KEY,
    spliced_key=SPLICED_KEY,
    velocity_key=VELOCITY_KEY,
    degradation_key=DEGRADATION_KEY,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    refit_gamma=True,
    w_scaffold=scaffold.values.T,
    scaffold_regularization=SCAFFOLD_REG,
    reconstruction_regularization=100,
    bias_regularization=1,
    only_TFs=True,
    w_threshold=1e-12,
    skip_all=True,
    learning_rate=0.1,
    use_plateau_scheduler=True,
    plateau_patience=100,
    plateau_factor=0.1,
    balanced_sampling=True,
    drop_last=True,
    include_neighbors=True,
    neighbor_fraction=0.2,
    get_plots=False,
)

clusters = list(adata.obs[CLUSTER_KEY].unique())
print(f"GRN inference complete for {len(clusters)} clusters.")


Inferring interaction matrix W and bias vector I for cluster 7MEP
  Including 402 neighboring cells (167 cluster + 402 neighbors = 569 total)


/home/bernaljp/packages/scHopfield/scHopfield/inference/optimizer.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('mask', torch.tensor(mask, dtype=torch.float32, device=device))
Training Epochs:   1%|          | 9/1000 [00:00<01:12, 13.62it/s]

[Epoch 1/1000] Total Loss: 609.155510, Reconstruction Loss: 91.877766, LR: 1.00e-01, Batch size: 128


Training Epochs:  11%|█▏        | 113/1000 [00:02<00:11, 77.17it/s]

[Epoch 101/1000] Total Loss: 139.680561, Reconstruction Loss: 0.456839, LR: 1.00e-01, Batch size: 128


...

[Epoch 801/1000] Total Loss: 1.288557, Reconstruction Loss: 0.007436, LR: 1.00e-03, Batch size: 30


Training Epochs:  97%|█████████▋| 970/1000 [00:02<00:00, 398.86it/s]

[Epoch 901/1000] Total Loss: 0.145170, Reconstruction Loss: 0.005128, LR: 1.00e-04, Batch size: 30


Training Epochs: 100%|██████████| 1000/1000 [00:02<00:00, 397.79it/s]

[Epoch 1000/1000] Total Loss: 0.132250, Reconstruction Loss: 0.007338, LR: 1.00e-04, Batch size: 30
GRN inference complete for 19 clusters.


In [9]:
# Colour map for consistent plotting (extracted from scVelo scatter)
fig_tmp, ax_tmp = plt.subplots()
scv.pl.scatter(adata, color=CLUSTER_KEY, basis=BASIS, ax=ax_tmp, show=False)
colors = {}
for k in adata.obs[CLUSTER_KEY].unique():
    idx = np.where(adata.obs[CLUSTER_KEY] == k)[0][0]
    c = ax_tmp.get_children()[0]._facecolors[idx].copy()
    c[3] = 1.0
    colors[k] = c
plt.close(fig_tmp)


 ## 5.2 ODE-based Single-Cell Trajectories



 Integrate the Hopfield ODE forward in time from a representative cell's initial

 state under wild-type, knockout, and overexpression conditions.

In [10]:
# Simulate WT, KO, and OE trajectories for the first three erythroid clusters
target_clusters = ['1Ery', '2Ery', '3Ery', '4Ery', '5Ery', '6Ery', '7MEP']
t_span = np.linspace(0, 10, 1000)

trajectory_results = {}

fig, axes = plt.subplots(min(3, len(target_clusters)), 2,
                         figsize=(14, 4 * min(3, len(target_clusters))),
                         tight_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]

for row, cluster in enumerate(target_clusters[:3]):
    print(f"  Simulating trajectories in {cluster}...")

    cluster_mask = adata.obs[CLUSTER_KEY] == cluster
    cluster_idx  = np.where(cluster_mask)[0]
    cell_idx     = cluster_idx[len(cluster_idx) // 2]  # representative cell

    wt = sch.dyn.simulate_trajectory(
        adata, cluster=cluster, spliced_key=SPLICED_KEY,
        cell_idx=cell_idx, t_span=t_span
    )
    ko = sch.dyn.simulate_perturbation_ode(
        adata, cluster=cluster, spliced_key=SPLICED_KEY,
        cell_idx=cell_idx, gene_perturbations={GOI: 0.0}, t_span=t_span
    )
    oe = sch.dyn.simulate_perturbation_ode(
        adata, cluster=cluster, spliced_key=SPLICED_KEY,
        cell_idx=cell_idx, gene_perturbations={GOI: 10.0}, t_span=t_span
    )
    trajectory_results[cluster] = {'wt': wt, 'ko': ko, 'oe': oe}

    # Top 5 most responsive genes (WT vs KO endpoint)
    top_idx    = np.argsort(np.abs(ko[-1] - wt[-1]))[-5:]
    clrs       = plt.cm.tab10(np.linspace(0, 1, len(top_idx)))

    for i, idx in enumerate(top_idx):
        axes[row, 0].plot(t_span, wt[:, idx], '-',  color=clrs[i], lw=2,
                          label=adata.var_names[idx])
        axes[row, 0].plot(t_span, ko[:, idx], '--', color=clrs[i], lw=2)
    axes[row, 0].set_xlabel('Time')
    axes[row, 0].set_ylabel('Expression')
    axes[row, 0].set_title(f'{cluster}: top responsive genes ({GOI} KO)\nSolid=WT  Dashed=KO')
    axes[row, 0].legend(fontsize=8)
    axes[row, 0].grid(True, alpha=0.3)

    # Phase-space divergence
    dist_wt = np.linalg.norm(wt - wt[0], axis=1)
    dist_ko = np.linalg.norm(ko - wt[0], axis=1)
    dist_oe = np.linalg.norm(oe - wt[0], axis=1)
    axes[row, 1].plot(t_span, dist_wt, color='gray',    lw=2, label='WT')
    axes[row, 1].plot(t_span, dist_ko, color='#E74C3C', lw=2, label=f'{GOI} KO')
    axes[row, 1].plot(t_span, dist_oe, color='#3498DB', lw=2, label=f'{GOI} OE')
    axes[row, 1].set_xlabel('Time')
    axes[row, 1].set_ylabel('Phase-space distance from t=0')
    axes[row, 1].set_title(f'{cluster}: trajectory divergence')
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.3)

plt.show()


  Simulating trajectories in 1Ery...
  Simulating trajectories in 2Ery...
  Simulating trajectories in 3Ery...


 ## 5.3 Dataset-wide ODE Perturbation



 Run the ODE simulation on every cell to obtain a perturbed expression landscape.

In [ ]:
print(f"Dataset-wide ODE simulation — {GOI} KO...")
adata_ode_ko = sch.dyn.simulate_shift_ode(
    adata.copy(),
    perturb_condition={GOI: 0.0},
    cluster_key=CLUSTER_KEY,
    dt=5.0,
    use_cluster_specific_GRN=True,
    verbose=True,
    n_jobs=-1,
)

print(f"Dataset-wide ODE simulation — {GOI} OE...")
adata_ode_oe = sch.dyn.simulate_shift_ode(
    adata.copy(),
    perturb_condition={GOI: 10.0},
    cluster_key=CLUSTER_KEY,
    dt=5.0,
    use_cluster_specific_GRN=True,
    verbose=True,
    n_jobs=-1,
)

top_ko = sch.dyn.get_top_affected_genes(adata_ode_ko, n_genes=20)
top_oe = sch.dyn.get_top_affected_genes(adata_ode_oe, n_genes=20)
print(f"Top ODE KO-affected genes: {top_ko[:5]}")
print(f"Top ODE OE-affected genes: {top_oe[:5]}")


Dataset-wide ODE simulation — Gata1 KO...
Processing cluster: 7MEP


Cells in 7MEP:   0%|          | 0/167 [00:00<?, ?it/s]

Processing cluster: 15Mo


Cells in 15Mo:   0%|          | 0/186 [00:00<?, ?it/s]

In [ ]:
# Perturbation effect heatmaps
fig_ko = sch.pl.plot_perturbation_effect_heatmap(adata_ode_ko, cluster_key=CLUSTER_KEY, n_genes=30)
plt.suptitle(f'{GOI} KO (ODE) — perturbation effect heatmap', y=1.01)
plt.show()

fig_oe = sch.pl.plot_perturbation_effect_heatmap(adata_ode_oe, cluster_key=CLUSTER_KEY, n_genes=30)
plt.suptitle(f'{GOI} OE (ODE) — perturbation effect heatmap', y=1.01)
plt.show()


In [ ]:
# Perturbation magnitude on embedding
sch.pl.plot_perturbation_magnitude(adata_ode_ko, cluster_key=CLUSTER_KEY, basis=BASIS)
plt.title(f'{GOI} KO (ODE) — perturbation magnitude')
plt.show()

sch.pl.plot_perturbation_magnitude(adata_ode_oe, cluster_key=CLUSTER_KEY, basis=BASIS)
plt.title(f'{GOI} OE (ODE) — perturbation magnitude')
plt.show()


In [ ]:
# Top affected genes (bar charts)
fig, axes = plt.subplots(1, 2, figsize=(16, 8), tight_layout=True)
sch.pl.plot_top_affected_genes_bar(adata_ode_ko, n_genes=20, ax=axes[0])
axes[0].set_title(f'{GOI} KO (ODE): top 20 affected genes')
sch.pl.plot_top_affected_genes_bar(adata_ode_oe, n_genes=20, ax=axes[1])
axes[1].set_title(f'{GOI} OE (ODE): top 20 affected genes')
plt.show()


In [ ]:
# Per-cluster KO vs OE symmetry and dual-column heatmaps
genes_used      = np.where(adata_ode_ko.var['scHopfield_used'])[0]
gene_names_used = adata_ode_ko.var_names[genes_used]

# Exclude the perturbed gene itself
perturb_keys = list(adata_ode_ko.uns['scHopfield'].get('perturb_condition', {}).keys())
keep = ~np.isin(gene_names_used, perturb_keys)
genes_used      = genes_used[keep]
gene_names_used = gene_names_used[keep]

delta_X_ko = np.asarray(adata_ode_ko.layers['delta_X'][:, genes_used])
delta_X_oe = np.asarray(adata_ode_oe.layers['delta_X'][:, genes_used])

cluster_changes_ko, cluster_changes_oe = {}, {}
for cl in clusters:
    mask = adata_ode_ko.obs[CLUSTER_KEY] == cl
    cluster_changes_ko[cl] = delta_X_ko[mask].mean(axis=0)
    cluster_changes_oe[cl] = delta_X_oe[mask].mean(axis=0)


In [ ]:
# Scatter: KO vs OE per-cluster symmetry
n_cl = len(clusters)
n_cols_plt = 4
n_rows_plt = (n_cl + n_cols_plt - 1) // n_cols_plt

fig, axes = plt.subplots(n_rows_plt, n_cols_plt,
                         figsize=(5 * n_cols_plt, 4 * n_rows_plt), tight_layout=True)
axes_flat = axes.flatten()

for i, cl in enumerate(clusters):
    ax  = axes_flat[i]
    m_ko = cluster_changes_ko[cl]
    m_oe = cluster_changes_oe[cl]
    t_ko = np.argsort(np.abs(m_ko))[-20:]
    t_oe = np.argsort(np.abs(m_oe))[-20:]
    ax.scatter(m_ko, m_oe, s=2, alpha=0.3, color='grey')
    ax.scatter(m_ko[t_ko], m_oe[t_ko], s=10, color='firebrick', label='Top KO')
    ax.scatter(m_ko[t_oe], m_oe[t_oe], s=10, color='steelblue', label='Top OE')
    lim = max(np.abs([ax.get_xlim(), ax.get_ylim()]).max(), 1e-9)
    ax.plot([-lim, lim], [lim, -lim], 'k--', alpha=0.2)
    ax.set_title(cl, fontsize=9)
    if i == 0:
        ax.legend(fontsize=7)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')
plt.suptitle(f'ODE KO vs OE symmetry per cluster ({GOI})', y=1.02)
plt.show()


In [ ]:
# Dual-column heatmap: top KO genes per cluster
fig, axes = plt.subplots(n_rows_plt, n_cols_plt,
                         figsize=(5 * n_cols_plt, 6 * n_rows_plt), tight_layout=True)
axes_flat = axes.flatten()

for i, cl in enumerate(clusters):
    ax   = axes_flat[i]
    m_ko = cluster_changes_ko[cl]
    m_oe = cluster_changes_oe[cl]
    idx  = np.argsort(np.abs(m_ko))[-20:][::-1]
    data = np.stack([m_ko[idx], m_oe[idx]], axis=1)
    v    = np.abs(data).max() or 1e-9
    ax.imshow(data, cmap='coolwarm', aspect='auto', vmin=-v, vmax=v)
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels(gene_names_used[idx], fontsize=7)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['KO', 'OE'], fontsize=8)
    ax.set_title(cl, fontsize=9)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')
plt.suptitle(f'Per-cluster top-KO genes — KO vs OE ({GOI})', y=1.02)
plt.show()


 ## 5.4 GRN Propagation-based Perturbation



 Propagate expression shifts through the inferred GRN iteratively

 (analogous to the CellOracle approach).

In [ ]:
N_PROPAGATION  = 3
DT             = 5.0

print(f"GRN propagation — {GOI} KO...")
adata_ko = sch.dyn.simulate_shift(
    adata.copy(),
    perturb_condition={GOI: 0.0},
    cluster_key=CLUSTER_KEY,
    n_propagation=N_PROPAGATION,
    use_cluster_specific_GRN=True,
    clip_delta_X=True,
    x_max_percentile=90.0,
    verbose=True,
    dt=DT,
)

print(f"GRN propagation — {GOI} OE...")
adata_oe = sch.dyn.simulate_shift(
    adata.copy(),
    perturb_condition={GOI: 10.0},
    cluster_key=CLUSTER_KEY,
    n_propagation=N_PROPAGATION,
    use_cluster_specific_GRN=True,
    clip_delta_X=True,
    x_max_percentile=90.0,
    verbose=True,
    dt=DT,
)

top_ko_prop = sch.dyn.get_top_affected_genes(adata_ko, n_genes=20)
top_oe_prop = sch.dyn.get_top_affected_genes(adata_oe, n_genes=20)
print(f"Propagation KO top genes: {top_ko_prop[:5]}")
print(f"Propagation OE top genes: {top_oe_prop[:5]}")


In [ ]:
# Heatmaps
fig_ko = sch.pl.plot_perturbation_effect_heatmap(adata_ko, cluster_key=CLUSTER_KEY, n_genes=30)
plt.suptitle(f'{GOI} KO (propagation) — perturbation effect heatmap', y=1.01)
plt.show()

fig_oe = sch.pl.plot_perturbation_effect_heatmap(adata_oe, cluster_key=CLUSTER_KEY, n_genes=30)
plt.suptitle(f'{GOI} OE (propagation) — perturbation effect heatmap', y=1.01)
plt.show()


In [ ]:
# Perturbation magnitude on embedding
sch.pl.plot_perturbation_magnitude(adata_ko, cluster_key=CLUSTER_KEY, basis=BASIS)
plt.title(f'{GOI} KO (propagation) — perturbation magnitude')
plt.show()

sch.pl.plot_perturbation_magnitude(adata_oe, cluster_key=CLUSTER_KEY, basis=BASIS)
plt.title(f'{GOI} OE (propagation) — perturbation magnitude')
plt.show()


In [ ]:
# Top affected genes (bar charts)
fig, axes = plt.subplots(1, 2, figsize=(16, 8), tight_layout=True)
sch.pl.plot_top_affected_genes_bar(adata_ko, n_genes=20, ax=axes[0])
axes[0].set_title(f'{GOI} KO (propagation): top 20 affected genes')
sch.pl.plot_top_affected_genes_bar(adata_oe, n_genes=20, ax=axes[1])
axes[1].set_title(f'{GOI} OE (propagation): top 20 affected genes')
plt.show()


 ## 5.5 Perturbation Flow on Embedding



 Project the perturbation-induced expression shifts onto the 2-D embedding

 using two approaches:



 * **CellOracle-style** (`method='celloracle'`) — correlation-based neighbour voting

 * **Hopfield-style** (`method='hopfield'`) — velocity derived directly from the

   model: Δv = v(x_perturbed) − v(x_original)

 ### CellOracle-style flow (propagation-based shift)

In [ ]:
n_neighbors_flow = 50

sch.tl.calculate_flow(
    adata_ko, basis=BASIS, method='celloracle',
    n_neighbors=n_neighbors_flow, correlation_mode='sampled', sigma_corr=0.05,
)
sch.tl.calculate_inner_product(
    adata_ko,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2=f'perturbation_flow_{BASIS}',
)

sch.tl.calculate_flow(
    adata_oe, basis=BASIS, method='celloracle',
    n_neighbors=n_neighbors_flow, correlation_mode='sampled', sigma_corr=0.05,
)
sch.tl.calculate_inner_product(
    adata_oe,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2=f'perturbation_flow_{BASIS}',
)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 15), tight_layout=True)

# Cell-type scatter
c = [colors.get(cl, 'gray') for cl in adata_ko.obs[CLUSTER_KEY]]
axes[0, 0].scatter(adata_ko.obsm[f'X_{BASIS}'][:, 0],
                   adata_ko.obsm[f'X_{BASIS}'][:, 1],
                   c=c, s=10, alpha=0.7)
axes[0, 0].set_title('Cell types')
axes[0, 0].axis('off')

# Reference velocity
sch.pl.plot_flow(
    adata, basis=BASIS, ax=axes[0, 1], on_grid=True,
    flow_key=f'velocity_S_{BASIS}',
    n_grid=25, min_mass=45, scale=1000, color='black',
    cluster_key=CLUSTER_KEY, colors=colors, title='Reference velocity (grid)',
)

sch.pl.plot_flow(
    adata_ko, basis=BASIS, ax=axes[1, 0], on_grid=True,
    flow_key=f'perturbation_flow_{BASIS}',
    n_grid=25, min_mass=25, scale=5, color='#E74C3C',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} KO — perturbation flow (grid)',
)
sch.pl.plot_flow(
    adata_oe, basis=BASIS, ax=axes[1, 1], on_grid=True,
    flow_key=f'perturbation_flow_{BASIS}',
    n_grid=25, min_mass=25, scale=5, color='#3498DB',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} OE — perturbation flow (grid)',
)

sch.pl.plot_inner_product(adata_ko, basis=BASIS, ax=axes[2, 0],
                          title=f'{GOI} KO — inner product')
sch.pl.plot_inner_product(adata_oe, basis=BASIS, ax=axes[2, 1],
                          title=f'{GOI} OE — inner product')

plt.show()


 ### Hopfield-style flow (ODE-based shift)

In [ ]:
n_neighbors_hopfield = 30

# KO: delta velocity (Δv = v' − v₀)
print(f"Computing Hopfield delta velocity — {GOI} KO...")
sch.tl.calculate_flow(
    adata_ode_ko, basis=BASIS, method='hopfield',
    cluster_key=CLUSTER_KEY, use_cluster_specific=True,
    source='delta', store_key='delta_velocity_hopfield',
    n_neighbors=n_neighbors_hopfield, n_jobs=-1, verbose=False,
)
sch.tl.calculate_inner_product(
    adata_ode_ko,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2='delta_velocity_hopfield',
    store_key='delta_inner_product_hopfield',
)

# KO: perturbed velocity (v')
print(f"Computing Hopfield perturbed velocity — {GOI} KO...")
sch.tl.calculate_flow(
    adata_ode_ko, basis=BASIS, method='hopfield',
    cluster_key=CLUSTER_KEY, use_cluster_specific=True,
    source='perturbed', store_key='perturbed_velocity_hopfield',
    n_neighbors=n_neighbors_hopfield, n_jobs=-1, verbose=False,
)
sch.tl.calculate_inner_product(
    adata_ode_ko,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2='perturbed_velocity_hopfield',
    store_key='perturbed_inner_product_hopfield',
)

# OE: delta velocity
print(f"Computing Hopfield delta velocity — {GOI} OE...")
sch.tl.calculate_flow(
    adata_ode_oe, basis=BASIS, method='hopfield',
    cluster_key=CLUSTER_KEY, use_cluster_specific=True,
    source='delta', store_key='delta_velocity_hopfield',
    n_neighbors=n_neighbors_hopfield, n_jobs=-1, verbose=False,
)
sch.tl.calculate_inner_product(
    adata_ode_oe,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2='delta_velocity_hopfield',
    store_key='delta_inner_product_hopfield',
)

# OE: perturbed velocity
print(f"Computing Hopfield perturbed velocity — {GOI} OE...")
sch.tl.calculate_flow(
    adata_ode_oe, basis=BASIS, method='hopfield',
    cluster_key=CLUSTER_KEY, use_cluster_specific=True,
    source='perturbed', store_key='perturbed_velocity_hopfield',
    n_neighbors=n_neighbors_hopfield, n_jobs=-1, verbose=False,
)
sch.tl.calculate_inner_product(
    adata_ode_oe,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2='perturbed_velocity_hopfield',
    store_key='perturbed_inner_product_hopfield',
)

print("Done computing all Hopfield flows.")


In [ ]:
# Visualise Hopfield delta and perturbed flow for KO
fig, axes = plt.subplots(2, 2, figsize=(16, 12), tight_layout=True)

sch.pl.plot_flow(
    adata_ode_ko, flow_key='delta_velocity_hopfield', basis=BASIS, on_grid=True,
    ax=axes[0, 0], n_grid=25, min_mass=25, scale=5000, color='#E74C3C',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} KO — Hopfield Δv (grid)',
)
sch.pl.plot_flow(
    adata_ode_ko, flow_key='perturbed_velocity_hopfield', basis=BASIS, on_grid=True,
    ax=axes[0, 1], n_grid=25, min_mass=25, scale=5000, color='#E74C3C',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f"{GOI} KO — Hopfield v' (grid)",
)
sch.pl.plot_inner_product(
    adata_ode_ko, basis=BASIS, ax=axes[1, 0],
    inner_product_key='delta_inner_product_hopfield',
    title=f'{GOI} KO — Hopfield Δv inner product',
)
sch.pl.plot_inner_product(
    adata_ode_ko, basis=BASIS, ax=axes[1, 1],
    inner_product_key='perturbed_inner_product_hopfield',
    title=f"{GOI} KO — Hopfield v' inner product",
)
plt.show()


In [ ]:
# Same for OE
fig, axes = plt.subplots(2, 2, figsize=(16, 12), tight_layout=True)

sch.pl.plot_flow(
    adata_ode_oe, flow_key='delta_velocity_hopfield', basis=BASIS, on_grid=True,
    ax=axes[0, 0], n_grid=25, min_mass=25, scale=5000, color='#3498DB',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} OE — Hopfield Δv (grid)',
)
sch.pl.plot_flow(
    adata_ode_oe, flow_key='perturbed_velocity_hopfield', basis=BASIS, on_grid=True,
    ax=axes[0, 1], n_grid=25, min_mass=25, scale=5000, color='#3498DB',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f"{GOI} OE — Hopfield v' (grid)",
)
sch.pl.plot_inner_product(
    adata_ode_oe, basis=BASIS, ax=axes[1, 0],
    inner_product_key='delta_inner_product_hopfield',
    title=f'{GOI} OE — Hopfield Δv inner product',
)
sch.pl.plot_inner_product(
    adata_ode_oe, basis=BASIS, ax=axes[1, 1],
    inner_product_key='perturbed_inner_product_hopfield',
    title=f"{GOI} OE — Hopfield v' inner product",
)
plt.show()
